# Лабораторная 3 Численные методы

Дан интеграл вида: $$\int^b_a f(x) dx$$ где $a = 0.35$, $b = 1.35$, $f(x)=0.35 e^x + 0.65 \cos x$.

Для вычисления интеграла с точностью $\varepsilon = 10^{-5}$ необходимо:

1. Пользуясь выражением для погрешности интегрирования, определить шаг $h$ в составной квадратурной формуле, которая обеспечит требуемую точность результата.
Рассмотреть квадратурная формулу Симпсона.
2. Для СКФ из п.1 определить величину $h$ шага разбиения исходного отрезка интегрирования, достаточного для достижения точности ε, по правилу Рунге.
3. Применить квадратурную формулу НАСТ Гаусса при указанном значении n. Оценить погрешность интегрирования через формулу остаточного члена $R_n(f)$. $n=1$.
4. Провести сравнительный анализ полученных в п.п.1-3 результатов.

### Аналитический вывод решения

$$\int^{1.35}_{0.35} 0.35 e^x + 0.65 \cos x = \left. (0.35 e^x + 0.65  \sin x) \right|_{0.35}^{1.35} \approx 1.264761901477586 $$

In [2]:
import numpy as np

def func(x):
    return 0.35 * np.exp(x) + 0.65 * np.cos(x)

a, b, eps = 0.35, 1.35, 10**(-5)

ANALYTICAL = 0.35 * (np.exp(1.35) - np.exp(0.35)) + 0.65 * (np.sin(1.35) - np.sin(0.35))
ANALYTICAL

np.float64(1.264761901477586)

### 1. Определение шага интегрирования при помощи выражения для погрешности

Для оценки количества разбиений отрезка воспользуемся формулой: $$N_{\text{сим}} \ge  \sqrt[4]{\frac{(b-a)^5 M_4}{180 \varepsilon}},\ \ M_4= \max |f^{(4)} (x)|$$

В контексте данной задачи: $$\begin{gathered} M_4 = \max |f^{(4)}(\xi) = 0.35 e^x + 0.65 \cos x|=1.49245, \\ N_{\text{сим}} \ge \sqrt[4]{\frac{1.49245}{180 \cdot 10^{-5}}} \end{gathered}$$

In [3]:
M = func(b)
R = (M / (180 * eps))
n_apr = R ** (1/4)
n_apr

np.float64(5.366078975029479)

Таким образом, $N_{\text{сим}} \ge 5.366$, значит, число разбиений, необходимое для достижения заданной точности, равняется $6$. Вычислим приближённое значение интеграла по квадратурной формуле Симпсона: $$I_{\text{Симп,С}} = \frac{h}{3} (f_0 + f_N + 2(f_2 + f_4 + \dots + f_{N-2}) + 4 (f_1 + f_3 + \dots + f_{N-1}))$$ И после вычислим невязку $|\int^b_a f(x) dx - I_{\text{Симп,С}}|$

In [4]:
N_1 = 6

def simpson(a, b, func, n):
    h = (b - a) / n
    nodes = np.linspace(a, b, n+1)
    y = func(nodes)
    return h/3 * (y[0] + y[-1] + 4*np.sum(y[1:-1:2]) + 2*np.sum(y[2:-2:2]))

I1 = simpson(a, b, func, N_1)
err1 = np.abs(ANALYTICAL - I1)
print(I1, "\nRedisual:", err1)

1.2647673169103477 
Redisual: 5.415432761779471e-06


Таким образом, приближённое значение интеграла: $1.2647673169103477$, невязка: $5.4154e-06$.

### 2. Определение шага по правилу Рунге

При помощи последовательного разбиения отрезка $[a,b]$ вычислим приближённое значение погрешности: $$R(h,f) \approx \frac{I_{h2} - I_{h1}}{1 - (\frac{h_2}{h_1})^m}$$ где $I_1$ - приближённое значение интеграла при разбиении с шагом $h_1$, $I_2$ - с шагом $h_2$.

Таким образом, для нахождения оптимльного числа разбиений отрезка, возьмём начальное значение $N$ - числа разбиений - и с каждой новой итерацией будем увеличивать его в два раза. В том случае, если $R(h,f) < \varepsilon$ (приближённое значение погрешности меньше априорной точности), останавливаем итерационный процесс и уточняем значение интеграла: $$I \approx I_{h1} + \frac{I_{h2} - I_{h1}}{1 - (\frac{h2}{h1})^m}$$ 

В качестве оптимального числа разбиений выберем то, при котором длина шага = $h_1$

В случае применения правила Рунге для формулы Симпсона $m=4$.

In [5]:
def runge_rule(formula, func, a, b):
    N_prev= 2
    I_prev = formula(a, b, func, N_prev)
    while True:
        N_next = N_prev * 2
        I_next = formula(a, b, func, N_next)
        R_runge = (I_next - I_prev) / (1 - (N_prev/N_next)**4)
        if np.abs(R_runge) < eps:
            break
        N_prev, I_prev = N_next, I_next
    I = I_prev + (I_next - I_prev) / (1 - (N_prev / N_next)**4)
    return I, N_prev
    
I_2, N_2 = runge_rule(simpson, func, a, b)
err2 = np.abs(I_2 - ANALYTICAL)
print(I_2, N_2)
print("Residual:", err2)

1.2647619015330274 8
Residual: 5.544142922531137e-11


Таким образом, вычисленное значение интеграла = $1.2647619015330274$ вычисленное число разбиений по правилу Рунге = $8$, невязка = $5.544e-11$

### 3. Применение квадратурной формулы НАСТ Гаусса

В контексте данной лабораторной работы будем рассматривать случай $p(x) = 1$ и $n=1$ и при помощи замены переменной $t = \frac{2x - a - b}{b-a}$ перейдём к отрезку $[-1,1]$. $$I = \int^1_{-1} f(x)dx \approx \sum^n_{k=0} A_k f(x_k)$$ В таком случае в качестве системы многочленов, ортогональых по весу $p(x)=1$ на отрезке $[-1,1]$ является системой многочленов Лежандра: $$P_n(x)=\frac{1}{2^n n!} \cdot \frac{d^n}{dx^n} (x^2 - 1)^n,\ n=0,1, \dots$$ И, поскольку $n=1$: $$\begin{gathered}P_1(x)=\frac{1}{2} \cdot (2x) = x \\ P_2(x)=\frac{3x^2-1}{2}\end{gathered}$$

Найдём узлы: $$P_2(x)=0 \Rightarrow x_0=-\frac{1}{\sqrt{3}},\ \ x_1=\frac{1}{\sqrt{3}}$$

В таком случае, учитывая свойства многочлена Лежандра, $$\begin{gathered}A_k=\frac{2}{(1-x^2_k)(P_{n+1}'(x_k))^2} \\ A_k =\frac{2}{(1-x^2_k)(3x_k)^2},\ \ k=\overline{0,n} \end{gathered}$$

Остаток квадратурной формулы имеет вид: $$R_1(f) \le \frac{32}{120}(\frac{4}{24})^2 f^4(\eta)$$ но это при интервале $[-1,1]$. Соответственно, после замены на $t$, заменим обратно на $x$, и тогда получится формула: $$R = \frac{b-a}{2} R_1 = \frac{1}{4320} \cdot 1.49245 = 3.454 \cdot 10^{-4}$$ 

In [ ]:
t_0, t_1 = -1/np.sqrt(3), 1/np.sqrt(3)
x_0, x_1 = ((b-a)*t_0 + b + a) / 2, ((b-a)*t_1 + b + a) / 2
A_0 = 2/(1 - t_0**2) / (3 * t_0)**2
A_1 = 2/(1 - t_1**2) / (3 * t_1)**2
I_3 = (b - a)/2 * (A_0 * func(x_0) + A_1 * func(x_1))


print("Приближённое значение I_3:", I_3)
print("Невязка:", I_3 - ANALYTICAL)

Приближённое значение I_3: 1.2644721364325902
Невязка: -0.00028976504499578226


Таким образом, вычисленное значение интеграла при помощи формулы НАСТ Гаусса = $1.2644721364325902$ по двум точкам, невязка = $-0.00029 = -2.9e-4$, оценка погрешности: $R=3.454e -4$.

### 4. Сравнительный анализ результатов

|Пункт|Число узлов|Невязка|
|--|--|--|
|1|7|$5.415e-06$|
|2|9|$5.544e-11$|
|3|2|$3.454e -4$|

**Анализ результатов:**  

1. Составная квадратурная формула Симпсона с шагом, определённым из априорной оценки погрешности (п. 1), обеспечила требуемую точность $\varepsilon = 10^{-5}$ при 7 узлах (фактическая невязка $\approx 5.4 \cdot 10^{-6}$). Это подтверждает корректность оценки.  

2. Уточнение шага по правилу Рунге (п. 2) позволило получить намного более точное значение интеграла (невязка порядка $10^{-11}$) при незначительном увеличении числа узлов (до 9). Это говорит о высокой эффективности правила Рунге для контроля точности. Также повлияло выполнение уточнения значения интеграла.  

3. Квадратурная формула Гаусса при $n=1$ (всего два узла) дала невязку $3.45 \cdot 10^{-4}$, что заметно больше $\varepsilon$. Однако такой результат полностью согласуется с теоретической оценкой остаточного члена ($\frac{(b-a)^5}{4320}\max|f^{(4)}| \approx 3.45 \cdot 10^{-4}$) и демонстрирует главное достоинство метода Гаусса: при одинаковом числе узлов он существенно точнее классических формул (например, погрешность формулы Симпсона на всём отрезке без разбиения составила бы порядка $10^{-3}$). Для достижения заданной точности потребовалось бы либо увеличить число узлов $n$, либо перейти к составной формуле Гаусса.
